# Call Red Online (HSE) APIs

Simple notebook — no project package imports. Uses standard library + optional `requests`.

On corporate networks, TLS verify is turned **off** (`verify=False`) to avoid `CERTIFICATE_VERIFY_FAILED`.

In [ ]:
import json
import ssl
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

# --- fill / confirm these ---
BASE_URL = "https://apigw.ct-test.hse-compliance.net"
API_KEY = "9f5ef52b-b98d-4cdb-9672-e423cc9051d5"
API_KEY_HEADER = "X-ROL-API-KEY"
SITE_ID = "112087"
USER_ID = "584646"
VERIFY_SSL = False  # False on corp laptop if SSL interception breaks certs

OUT_DIR = Path("../samples/rol")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENDPOINTS = {
    "list_sites": {
        "path": "/v2/secure-clients/legapi-general/sites",
        "params": {},
    },
    "list_users_by_site": {
        "path": f"/v2/secure-clients/legapi-general/users/sites/{SITE_ID}",
        "params": {},
    },
    "list_mapping_references": {
        "path": "/v2/secure-clients/tasks/configs/references",
        "params": {"user_id": USER_ID},
    },
    "list_tasks_by_user": {
        "path": "/v2/secure-clients/tasks",
        "params": {"user_id": USER_ID},
    },
}

In [ ]:
def rol_get(path: str, params: dict | None = None):
    url = BASE_URL.rstrip("/") + path
    if params:
        url = f"{url}?{urlencode(params)}"
    headers = {
        "Accept": "application/json",
        API_KEY_HEADER: API_KEY,
    }
    req = Request(url, headers=headers, method="GET")
    context = None if VERIFY_SSL else ssl._create_unverified_context()
    print("GET", url)
    with urlopen(req, timeout=60, context=context) as resp:
        body = resp.read().decode("utf-8")
        return json.loads(body) if body else None


def save_json(name: str, payload):
    path = OUT_DIR / f"{name}_raw.json"
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    print("saved", path.resolve())
    return path

## Call one endpoint (sites)

In [ ]:
sites = rol_get(ENDPOINTS["list_sites"]["path"])
save_json("list_sites", sites)
sites if not isinstance(sites, list) else sites[:2]

## Call all endpoints and save responses

In [ ]:
results = {}
for name, cfg in ENDPOINTS.items():
    try:
        payload = rol_get(cfg["path"], cfg.get("params") or {})
        save_json(name, payload)
        results[name] = {
            "ok": True,
            "type": type(payload).__name__,
            "count": len(payload) if isinstance(payload, list) else None,
            "top_keys": list(payload.keys())[:20] if isinstance(payload, dict) else None,
        }
    except Exception as exc:
        results[name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}"}
        print("FAILED", name, results[name]["error"])

results

## Inspect keys for RAW table design

If the payload is a list of objects, these keys become candidate RAW columns.

In [ ]:
def first_object(payload):
    if isinstance(payload, list) and payload and isinstance(payload[0], dict):
        return payload[0]
    if isinstance(payload, dict):
        for key in ("data", "items", "results", "value"):
            val = payload.get(key)
            if isinstance(val, list) and val and isinstance(val[0], dict):
                return val[0]
            if isinstance(val, dict):
                return val
        return payload
    return None


for name in ENDPOINTS:
    path = OUT_DIR / f"{name}_raw.json"
    if not path.exists():
        print(name, "-> no saved file")
        continue
    payload = json.loads(path.read_text(encoding="utf-8"))
    sample = first_object(payload)
    print("\n===", name, "===")
    if sample is None:
        print("could not find object sample; payload type:", type(payload).__name__)
    else:
        for key, value in sample.items():
            print(f"  {key}: {type(value).__name__} = {repr(value)[:80]}")